# Notebook 2: Building your own pipeline with QSARmil's building blocks

`01_Beginner_Easy_Start.ipynb` showed how `MultiConformerRegressor`/`MultiConformerClassifier` do everything for
you automatically. This notebook opens up that black box: it walks through each step separately (generating 3D
shapes, computing descriptors, training a model, making predictions), explains exactly what data goes in and comes
out of each step, and shows every setting you can change along the way.

This is the notebook to read if you want to:
- plug in your own way of generating 3D shapes (conformers),
- plug in your own descriptor,
- understand exactly what each model setting does, or
- benchmark different combinations yourself instead of letting QSARmil search automatically.

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from sklearn.metrics import r2_score

### 1. Load data

Same small activity dataset as in Notebook 1.

In [ ]:
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]].reset_index(drop=True)
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]].reset_index(drop=True)

Since this notebook is about exploring the pipeline rather than getting the best possible score, we work with a
small random sample throughout - this keeps every cell fast to re-run while you experiment. Feel free to increase
the sample size (or use the full dataset) once you're happy with your setup.

In [ ]:
df_train = df_train.sample(n=15, random_state=42).reset_index(drop=True)
df_test = df_test.sample(n=5, random_state=42).reset_index(drop=True)
df_train.shape, df_test.shape

### 2. Generating 3D shapes (conformers)

A single 2D structure (a SMILES string) can fold into several different 3D shapes. QSARmil treats each molecule as
a **bag** of these 3D shapes (called **conformers**), and each conformer in the bag is one **instance**. This is
the "multi-instance" part of multi-instance learning: instead of one fixed 3D structure per molecule, we keep
several and let the model decide how to combine them.

**Input:** a plain list of RDKit `Mol` objects (one per molecule - not yet embedded in 3D).

**Output:** a list with one entry per input molecule, in the same order. Each entry is either:
- a plain Python `list` of single-conformer `Mol` objects (the "bag" for that molecule), if 3D generation
  succeeded, or
- a `FailedMolecule` (SMILES couldn't be parsed) or `FailedConformer` (parsed fine, but 3D embedding failed)
  object, if it didn't.

Nothing raises an error when a molecule fails - a bad molecule just becomes one of these two sentinel objects
instead of a bag, so a handful of problem structures don't stop the whole batch. QSARmil's higher-level classes
(like `MultiConformerRegressor`) check for these automatically and report which molecules were dropped; if you're
calling the generator directly like below, it's up to you to check for them before moving on.

QSARmil's built-in generator, `RDKitConformerGenerator`, uses RDKit's ETKDG method to embed the 3D shapes and then
UFF to relax (optimize) them. Its settings are:

| Parameter | What it means | Default |
|---|---|---|
| `num_conf` | Maximum number of 3D shapes to generate per molecule. | `10` |
| `e_thresh` | Energy cutoff, relative to the most stable shape found. Any shape more than this much less stable is thrown away. `None` disables this filtering. | `None` |
| `num_cpu` | Number of CPU threads to use. | `1` |
| `verbose` | Whether to print a progress counter. | `True` |
| `seed` | Random seed for 3D embedding, for reproducible shapes. | `42` |

If you want to use your **own** conformer generator (e.g. from a different tool, or pre-computed structures you
already have), it just needs to look like the above: something with a `.run(list_of_mols)` method that returns a
list of plain `list[Mol]` bags in the same order as its input. If you already have RDKit `Mol` objects that already
contain multiple embedded conformers (e.g. loaded from a file), use `qsarmil.conformer.split_into_conformers(mol)`
to turn one of them into the `list[Mol]` bag shape the rest of the pipeline expects.

In [ ]:
from qsarmil.conformer import RDKitConformerGenerator

conf_gen = RDKitConformerGenerator(num_conf=5, e_thresh=50, num_cpu=2, verbose=True)

In [ ]:
mols_train = [Chem.MolFromSmiles(smi) for smi in df_train["smiles"]]
mols_test = [Chem.MolFromSmiles(smi) for smi in df_test["smiles"]]

confs_train = conf_gen.run(mols_train)
confs_test = conf_gen.run(mols_test)

# one bag = a plain list of single-conformer Mol objects, one per generated 3D shape
bag = confs_train[0]
print(type(bag), "with", len(bag), "conformers")

### 3. Computing descriptors

3D shapes aren't numbers yet, so a model can't learn from them directly. A **descriptor** turns one conformer into
a fixed-size list of numbers. Since each molecule is a bag of several conformers, the result is a bag of descriptor
vectors: one row per conformer.

QSARmil's `DescriptorWrapper` handles this for you: it takes any descriptor calculator (from RDKit, from MolFeat,
or your own) and applies it to every conformer of every molecule.

**Input to `DescriptorWrapper.run(...)`:** the list of bags produced in the previous step (only real bags - drop
any `FailedMolecule`/`FailedConformer` entries first).

**Output:** a list of 2D NumPy arrays, one per molecule, shaped `(number of conformers, number of features)`.

To use your **own** descriptor, all it needs to be is a callable that takes one conformer (a single-conformer
`Mol`) and returns a 1D NumPy array of numbers - i.e. something with the shape `transformer(mol, conformer_id=0) ->
np.ndarray`. Wrap it in `DescriptorWrapper(my_transformer)` and it works exactly like the built-in ones below.

QSARmil ships several ready-to-use 3D descriptor types:
- **RDKit-based:** `RDKitGEOM`, `RDKitAUTOCORR`, `RDKitRDF`, `RDKitMORSE`, `RDKitWHIM`, `RDKitGETAWAY`
- **MolFeat-based:** `Pharmacophore3D`, `USRDescriptors`, `ElectroShapeDescriptors`

Below, we compute one descriptor type (`Pharmacophore3D`) as an example - repeat the same lines for any other
descriptor type to compute more than one (this is exactly what QSARmil's automatic pipeline does internally with
all nine).

In [ ]:
from molfeat.calc import Pharmacophore3D
from qsarmil.descriptor.wrapper import DescriptorWrapper

desc_calc = DescriptorWrapper(Pharmacophore3D(factory="pmapper"), verbose=True)

In [ ]:
x_train = desc_calc.run(confs_train)
x_test = desc_calc.run(confs_test)

One last step before training: descriptor values are usually on very different scales (some might range in the
thousands, others between -1 and 1), which makes training harder. `BagMinMaxScaler` rescales every column to the
same `[0, 1]` range - fit it on the training data only, then reuse that same fit to transform both splits.

In [ ]:
from milearn.preprocessing import BagMinMaxScaler

scaler = BagMinMaxScaler()
scaler.fit(x_train)

x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

y_train, y_test = df_train["y"], df_test["y"]

### 4. Choosing and training a multi-instance model

A regular machine learning model expects one fixed-size vector per example. Here, each molecule is a *bag* of
several conformer vectors instead - a multi-instance learning (MIL) model is what knows how to combine them into
one prediction. QSARmil (via the `milearn` package) offers a few different families of these:

- **Wrapper networks** - take a normal neural network and bolt a simple combination rule onto it.
  `BagWrapperMLPNetwork` combines the conformers first, then predicts; `InstanceWrapperMLPNetwork` predicts for
  every conformer first, then combines the predictions.
- **Classic MIL networks** (`BagNetwork`, `InstanceNetwork`) - similar idea, but the combination step is built
  directly into the network instead of wrapped around it.
- **Attention-based networks** (`AdditiveAttentionNetwork`, `SelfAttentionNetwork`, `HopfieldAttentionNetwork`) -
  the network *learns* how much weight to give each conformer, instead of using a fixed rule like "average them
  all". This is what makes key instance detection possible - see `03_Key_Instance_Detection.ipynb`.
- **`DynamicPoolingNetwork`** - another attention-style network with a different (iterative) way of computing
  those weights.

Every one of these accepts the same core settings (they all build on the same underlying `BaseNetwork`):

| Parameter | What it means | Default |
|---|---|---|
| `hidden_layer_sizes` | Sizes of the hidden layers in the network, e.g. `(256, 128, 64)` means 3 layers of those sizes. | `(256, 128, 64)` |
| `max_epochs` | Maximum number of passes over the training data. | `1000` |
| `batch_size` | Number of molecules processed together in one training step. | `128` |
| `activation` | The activation function used between layers (e.g. `"relu"`, `"gelu"`, `"elu"`, `"silu"`, `"leakyrelu"`). | `"gelu"` |
| `learning_rate` | How big a step the optimizer takes at each update. Smaller is more cautious but slower. | `0.001` |
| `early_stopping` | Whether to stop training automatically once the model stops improving, instead of always running for `max_epochs`. | `True` |
| `weight_decay` | A regularization strength that discourages overly large weights (helps avoid overfitting). | `0.0` |
| `instance_dropout` | Fraction of conformers randomly ignored during each training step, as another way to avoid overfitting. | `0.0` |
| `accelerator` | `"cpu"` or `"gpu"` - which hardware to train on. | `"cpu"` |
| `verbose` | Whether to print training progress. | `False` |
| `random_seed` | Fixed random seed, for reproducible training. | `42` |
| `num_workers` | Number of background workers used to load data during training. | `0` |

A few network families take one extra setting on top of these:
- `pool` (on `BagNetwork`/`InstanceNetwork`/the wrapper networks) - how conformers are combined: `"mean"`, `"sum"`,
  `"max"`, or `"lse"`.
- `tau` (on the attention-based networks) - a temperature that controls how "sharp" or "soft" the learned
  attention weights are; smaller values push the network toward focusing on fewer conformers.

Let's train one, spelling out its settings explicitly:

In [ ]:
from milearn.network.regressor import AdditiveAttentionNetworkRegressor

model = AdditiveAttentionNetworkRegressor(
    hidden_layer_sizes=(256, 128, 64),  # 3 hidden layers of these sizes
    max_epochs=100,                      # small, so this cell runs quickly on the sample data
    batch_size=128,
    activation="gelu",
    learning_rate=0.001,
    early_stopping=True,
    weight_decay=0.0,
    instance_dropout=0.0,
    accelerator="cpu",
    verbose=False,
    random_seed=42,
    num_workers=0,
    tau=1.0,                             # extra setting specific to attention-based networks
)
model.fit(x_train_scaled, y_train)

### 5. Automatic hyperparameter search (optional)

Instead of choosing every setting above by hand, you can let `milearn` search for good values itself, one setting
at a time. This is what QSARmil's `hopt=True` option (see Notebook 1) does behind the scenes for every model it
trains.

This can take a while (it trains many candidate models), so we use a small `max_epochs` here to keep this example
notebook fast - use a larger value for real runs.

In [ ]:
from milearn.network.module.hopt import DEFAULT_PARAM_GRID

# default search grid - each list is a set of candidate values to try for that setting
DEFAULT_PARAM_GRID

In [ ]:
quick_grid = {**DEFAULT_PARAM_GRID, "max_epochs": 30}  # smaller max_epochs, just for this notebook

model = AdditiveAttentionNetworkRegressor()
model.hopt(x_train_scaled, y_train, param_grid=quick_grid, verbose=True)
model.fit(x_train_scaled, y_train)

### 6. Predicting on new molecules

Once trained, a model predicts the usual way. Attention-based and `DynamicPoolingNetwork` models additionally
offer `get_instance_weights(x)`, which returns the weight the model assigned to *each conformer* - i.e. how much
each 3D shape contributed to the final prediction. We won't use it here, but it's exactly what
`03_Key_Instance_Detection.ipynb` explores in depth.

In [ ]:
y_pred = model.predict(x_test_scaled)
w_pred = model.get_instance_weights(x_test_scaled)

r2_score(y_test, y_pred)

### What's next?

- `01_Beginner_Easy_Start.ipynb` shows how all of the steps above are automated for you.
- `03_Key_Instance_Detection.ipynb` shows what to do with `get_instance_weights` - finding out *which* conformer a
  model thinks matters most.